In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages
import re
import os
print("Libraries Imported")

Libraries Imported


In [2]:
from utils import format_signal, format_flow, format_sleep_profile

In [3]:
#returned a list
events = format_flow("Data/AP01/flow_events.txt")
events[0:3]

[{'start': Timestamp('2024-05-30 23:48:45.119000'),
  'end': Timestamp('2024-05-30 23:49:01.408000'),
  'duration': 16,
  'label': 'Hypopnea',
  'stage': 'N1'},
 {'start': Timestamp('2024-05-30 23:50:16.578000'),
  'end': Timestamp('2024-05-30 23:50:33.546000'),
  'duration': 17,
  'label': 'Hypopnea',
  'stage': 'N1'},
 {'start': Timestamp('2024-05-30 23:52:13.626000'),
  'end': Timestamp('2024-05-30 23:52:27.268000'),
  'duration': 14,
  'label': 'Hypopnea',
  'stage': 'N1'}]

In [4]:
subject_folder=input("Enter the Subject folder path")
if not os.path.exists(subject_folder):
        print("Folder does not exist")
else:
    flow_df=format_signal(f"{subject_folder}/nasal_airflow.txt")
    thorac_df=format_signal(f"{subject_folder}/thoracic_movement.txt")
    spo2_df=format_signal(f"{subject_folder}/spo2.txt")
    events=format_flow(f"{subject_folder}/flow_events.txt") #dict
    sleep_df=format_sleep_profile(f"{subject_folder}/sleep_profile.txt")

    print("All files are loaded!")

Enter the Subject folder path Data/AP02


All files are loaded!


In [5]:
'''returns a df
spo2_df=format_signal("Data/AP01/spo2.txt")
spo2_df.head()'''

'returns a df\nspo2_df=format_signal("Data/AP01/spo2.txt")\nspo2_df.head()'

In [6]:
def plot_signal(ax, df, color, ylabel, title): #signal
    t0=df.index[0]
    hours=(df.index-t0).total_seconds()/3600

    ax.plot(
        hours,
        df["value"].values,
        color=color,
        linewidth=0.5,
        alpha=0.9
    )

    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3, linewidth=0.4)

    return t0, hours

In [7]:
#flow_events
event_colours={
    "Hypopnea" : "#FDFF00", #yellow
    "Apnea" : "#FF8400" #orange
}

def shade_event(ax, events, t0): #flow_events
    for ev in events:
        start = (ev["start"]-t0).total_seconds()/3600
        end= (ev["end"]-t0).total_seconds()/3600
        colour=event_colours.get(ev["label"],"#00FF2F") #green as default for unknown
        ax.axvspan(
            start, end,
            alpha=0.2,
            color=colour
        )

In [8]:
u_stages = set(item['stage'] for item in events)
print(u_stages)

{'N3', 'N1', 'Wake', 'REM', 'N2'}


In [9]:
sleep_df =format_sleep_profile("Data/AP01/sleep_profile.txt")
print(sleep_df["stage"].unique())

['Wake' 'N1' 'N2' 'N3' 'REM']


In [12]:
#sleep_profiles
#pastel colours sleep profile
stage_colours={ 
    "Wake":"#FF775C",
    "REM":"#FFB65C",
    "N1": "#BDFF91",
    "N2": "#8AFFF3",
    "N3":"#8A9DFF"
}

def plot_sleep_profile(px, sleep_df, t0):
    stages=sleep_df["stage"].values #np
    t=sleep_df.index
    h=(t-t0).total_seconds()/3600
    s=len(stages)
    for i in range(s-1):
        colour=stage_colours.get(stages[i], "#AAAAAA") #default grey
        px.barh(
            0,                          
            h[i+1] - h[i],
            left=h[i],
            color=colour,
            height=1,
            align="center"
        )
    px.set_ylabel("Sleep Stage", fontsize=8)
    px.set_title("Sleep Stages", fontsize=10, fontweight="bold")
    px.tick_params(labelsize=7)
    # create a colored box for each sleep stage
    patches = [
        mpatches.Patch(color=colour, label=stages)
        for stage, colour in stage_colours.items()
    ]

    # add legend to plot
    px.legend(
        handles=patches,
        loc="upper right",
        fontsize=6,
        ncol=4,
        framealpha=0.7
    )

In [13]:
fig, axes = plt.subplots(
    4, 1,                                          # 4 rows, 1 column
    figsize=(18, 14),                              # width=18, height=14 inches
    sharex=True,                                   # all plots share same x axis
    gridspec_kw={"height_ratios": [3, 3, 2, 1]}   # plot heights — signal plots bigger, sleep stage smaller
)

# main title
subject = os.path.basename(subject_folder)  # extracts "AP01" from "Data/AP01"
fig.suptitle(
    f"Sleep Study Signals - Participant: {subject}",
    fontsize=12,
    fontweight="bold",
    y=0.98
)

#1. Nasal Airflow
t0, hours = plot_signal(
    axes[0],
    flow_df,
    "#2471A3", #blue
    "Amplitude",
    "Nasal Airflow (32 Hz)"
)
shade_event(axes[0], events, t0)   # shade events on top plot

#2. Thoracic Movement
plot_signal(
    axes[1],
    thorac_df,
    "#1E8449",#green
    "Amplitude",
    "Thoracic Movement (32 Hz)"
)
shade_event(axes[1], events, t0)
plot_signal(
    axes[2],
    spo2_df,
    "#CB4335", #red
    "SpO2 (%)",
    "Oxygen Saturation - SpO2 (4 Hz)"
)

axes[2].set_ylim(70, 100)             
shade_event(axes[2], events, t0)

#3. Sleep Stages
plot_sleep_profile(axes[3], sleep_df, t0)

axes[3].set_xlabel("Time (hours from recording start)", fontsize=9)

patches = [
    mpatches.Patch(color="#FFA500", label="Hypopnea"),
    mpatches.Patch(color="#FF3333", label="Obstructive Apnea"),
]
fig.legend(
    handles=patches,
    loc="lower center",
    fontsize=7,
    ncol=2,
    title="Breathing Events",
    title_fontsize=7,
    bbox_to_anchor=(0.5, 0.01),
    framealpha=0.8
)

plt.tight_layout(rect=[0, 0.05, 1, 0.97])

#Saving as PDF
os.makedirs("Visualizations", exist_ok=True)   # create folder if it doesn't exist
out_path = f"Visualizations/{subject}_visualization.pdf"

with PdfPages(out_path) as pdf:
    pdf.savefig(fig, bbox_inches="tight")
    d = pdf.infodict()
    d["Title"]  = f"Sleep Study – {subject}"
    d["Author"] = "SRIP 2026"

plt.close()   # close the plot to free memory
print(f"✅ PDF saved → {out_path}")

C:\Users\ANJANA\AppData\Local\Temp\ipykernel_3000\1660297544.py:67: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout(rect=[0, 0.05, 1, 0.97])


PDF saved → Visualizations/AP02_visualization.pdf
